# Demo 1 — Optimization pipelines: equity and Polymarket

This notebook is a compact, fully runnable demo of the project’s optimization layer. It covers:

1. **Equity portfolio optimization** on the included S&P 500 monthly panel.
2. **Polymarket mispricing optimization** on a small daily-panel subset using the reusable `src/polymarket` modules.

The goal is to show the core objective, constraints, and key output tables without needing external downloads or manual uploads.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root whether this notebook is run from root or notebooks/."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing src/ and data/.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / "Outputs" / "demo_optimization"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
np.random.seed(42)

# Import Polymarket/sklearn-dependent modules before the torch-based equity optimizer runs.
# This avoids occasional OpenMP/threading stalls in some notebook kernels.
from polymarket.backtest import MispricingConfig, run_walk_forward
from polymarket.diagnostics import plot_run
from polymarket.load import enrich_snapshot

print("Repo root:", ROOT)
print("Output directory:", OUT)

## Part A — Equity portfolio optimization

The equity optimizer follows the objective used in the course notebooks:

\[
\max_w \mu_t^\top w - \gamma w^\top \Sigma_t w - \kappa \lVert w - w_{t-1} \rVert_1
\]

subject to long-only simplex weights. At each rebalance date, the demo estimates `mu` and `Sigma` from a rolling historical window, aligns the previous weights to the current universe, optimizes, and realizes next-month returns. The demo intentionally uses a smaller top-market-cap universe so it runs quickly.

In [ ]:
from src.model import estimate_moments, optimize_weights

EQUITY_CSV = ROOT / "data" / "stock_market" / "sp500_monthly (1) (1).csv"
assert EQUITY_CSV.is_file(), f"Missing {EQUITY_CSV}"

df_eq = pd.read_csv(EQUITY_CSV)
df_eq["date"] = pd.to_datetime(df_eq["date"], errors="coerce")
df_eq = df_eq.sort_values(["date", "permno"]).copy()

# CRSP-style common-share/main-exchange filter used in the project notebooks.
if "shrcd" in df_eq.columns:
    df_eq = df_eq[df_eq["shrcd"].isin([10, 11])].copy()
if "exchcd" in df_eq.columns:
    df_eq = df_eq[df_eq["exchcd"].isin([1, 2, 3])].copy()

df_eq["ret"] = pd.to_numeric(df_eq["ret"], errors="coerce")
df_eq["prc"] = pd.to_numeric(df_eq["prc"], errors="coerce").abs()
df_eq["shrout"] = pd.to_numeric(df_eq["shrout"], errors="coerce")
df_eq["mcap"] = df_eq["prc"] * df_eq["shrout"]
df_eq = df_eq.dropna(subset=["date", "permno", "ret"]).copy()

ret_panel = df_eq.pivot_table(index="date", columns="permno", values="ret", aggfunc="first").sort_index()
mcap_panel = df_eq.pivot_table(index="date", columns="permno", values="mcap", aggfunc="first").sort_index()

equity_summary = {
    "rows": len(df_eq),
    "unique_assets": int(df_eq["permno"].nunique()),
    "unique_months": int(df_eq["date"].nunique()),
    "date_min": str(df_eq["date"].min().date()),
    "date_max": str(df_eq["date"].max().date()),
    "return_panel_shape": ret_panel.shape,
}
display(pd.DataFrame([equity_summary]))

In [ ]:
def align_previous_weights(current_assets: list, w_prev_dict: dict | None) -> np.ndarray:
    if w_prev_dict is None:
        return np.ones(len(current_assets)) / len(current_assets)
    w = np.array([w_prev_dict.get(asset, 0.0) for asset in current_assets], dtype=float)
    return w / w.sum() if w.sum() > 0 else np.ones(len(current_assets)) / len(current_assets)


def performance_metrics(returns: np.ndarray, freq: int = 12) -> dict:
    returns = np.asarray(returns, dtype=float)
    if len(returns) == 0:
        return {"mean_period": np.nan, "vol_annual": np.nan, "sharpe_annual": np.nan, "max_drawdown": np.nan, "cum_return": np.nan}
    equity = np.cumprod(1.0 + returns)
    running_max = np.maximum.accumulate(equity)
    drawdown = equity / running_max - 1.0
    std = returns.std(ddof=1) if len(returns) > 1 else np.nan
    return {
        "mean_period": float(returns.mean()),
        "vol_annual": float(std * np.sqrt(freq)) if std and not np.isnan(std) else np.nan,
        "sharpe_annual": float(np.sqrt(freq) * returns.mean() / std) if std and std > 0 else np.nan,
        "max_drawdown": float(drawdown.min()),
        "cum_return": float(equity[-1] - 1.0),
    }


def run_equity_backtest(
    ret_panel: pd.DataFrame,
    mcap_panel: pd.DataFrame,
    rolling_window: int = 24,
    min_history: int = 36,
    k_universe: int = 60,
    months_to_run: int = 96,
    gamma: float = 5.0,
    kappa: float = 0.10,
    steps: int = 80,
    lr: float = 0.08,
) -> dict:
    dates = list(ret_panel.index)
    start_idx = max(rolling_window + min_history, len(dates) - months_to_run)
    hist_count = ret_panel.notna().rolling(min_history).sum().shift(1)

    port_returns, bench_returns, bt_dates, turnovers = [], [], [], []
    weight_history = []
    w_prev_dict = None

    for t_idx in range(start_idx, len(dates) - 1):
        date_t, date_next = dates[t_idx], dates[t_idx + 1]

        eligible = hist_count.loc[date_t]
        eligible = eligible[eligible >= min_history].index
        mcap_t = mcap_panel.loc[date_t, eligible].dropna().sort_values(ascending=False)
        current_assets = list(mcap_t.head(k_universe).index)
        if len(current_assets) < 5:
            continue

        window = ret_panel.loc[dates[t_idx - rolling_window:t_idx], current_assets]
        valid_assets = list(window.columns[window.notna().all(axis=0)])
        if len(valid_assets) < 5:
            continue
        window = window[valid_assets]
        current_assets = valid_assets

        r_next = ret_panel.loc[date_next, current_assets].fillna(0.0).to_numpy(dtype=float)
        mu, Sigma = estimate_moments(window)
        w_prev = align_previous_weights(current_assets, w_prev_dict)
        w = optimize_weights(mu, Sigma, w_prev=w_prev, gamma=gamma, kappa=kappa, steps=steps, lr=lr, device="cpu")

        port_returns.append(float(w @ r_next))
        bench_returns.append(float(np.mean(r_next)))
        turnovers.append(float(np.abs(w - w_prev).sum()))
        bt_dates.append(date_next)
        weight_history.append((date_t, current_assets, w))
        w_prev_dict = dict(zip(current_assets, w))

    return {
        "dates": pd.DatetimeIndex(bt_dates),
        "portfolio_returns": np.array(port_returns),
        "benchmark_returns": np.array(bench_returns),
        "turnover": np.array(turnovers),
        "weight_history": weight_history,
        "config": {
            "rolling_window": rolling_window,
            "min_history": min_history,
            "k_universe": k_universe,
            "months_to_run": months_to_run,
            "gamma": gamma,
            "kappa": kappa,
            "steps": steps,
            "lr": lr,
        },
    }


eq_result = run_equity_backtest(ret_panel, mcap_panel)
metrics = pd.DataFrame(
    [
        {"strategy": "optimized", **performance_metrics(eq_result["portfolio_returns"]), "avg_turnover": float(eq_result["turnover"].mean())},
        {"strategy": "equal_weight_same_universe", **performance_metrics(eq_result["benchmark_returns"]), "avg_turnover": np.nan},
    ]
)
display(metrics.round(4))
print("Backtest months:", len(eq_result["dates"]))
print("Config:", eq_result["config"])

In [ ]:
eq_curve = pd.DataFrame(
    {
        "optimized": np.cumprod(1 + eq_result["portfolio_returns"]),
        "equal_weight_same_universe": np.cumprod(1 + eq_result["benchmark_returns"]),
    },
    index=eq_result["dates"],
)

ax = eq_curve.plot(figsize=(9, 4), title="Equity demo: growth of $1")
ax.set_xlabel("date")
ax.set_ylabel("growth of $1")
plt.tight_layout()
plt.show()

# Show average holdings to verify this is not an equal-weight mechanical result.
all_assets = sorted({asset for _, assets, _ in eq_result["weight_history"] for asset in assets})
wmat = pd.DataFrame(0.0, index=[d for d, _, _ in eq_result["weight_history"]], columns=all_assets)
for d, assets, w in eq_result["weight_history"]:
    wmat.loc[d, assets] = w
weight_summary = wmat.mean().sort_values(ascending=False).head(10).to_frame("avg_weight")
display(weight_summary)
print("Average L1 distance from equal-weight among held universe is nonzero by construction; turnover is shown in the metrics table.")

## Part B — Polymarket mispricing optimization

The Polymarket optimizer estimates a model-implied probability `p_hat`, compares it to an executable Yes price, and chooses long Yes weights with risk, turnover, concentration, and liquidity constraints.

The objective implemented in `src/polymarket/optimizer.py` is:

\[
F(w)=\sum_i w_i (\hat p_i-c_i) - \gamma \sum_i w_i^2 \hat p_i(1-\hat p_i) - \kappa \lVert w-w_{t-1}\rVert_1.
\]

This demo runs the existing walk-forward pipeline on a small, chronologically ordered panel subset so it finishes quickly.

In [ ]:
POLY_PANEL_CSV = ROOT / "data" / "prediction_market" / "polymarket_daily_panel_60plus.csv"
assert POLY_PANEL_CSV.is_file(), f"Missing {POLY_PANEL_CSV}"

raw_poly = pd.read_csv(POLY_PANEL_CSV, low_memory=False)
raw_poly["date"] = pd.to_datetime(raw_poly["date"], utc=True, errors="coerce")
raw_poly["price"] = pd.to_numeric(raw_poly["price"], errors="coerce")
raw_poly["liquidity"] = pd.to_numeric(raw_poly["liquidity"], errors="coerce")
raw_poly["volume"] = pd.to_numeric(raw_poly["volume"], errors="coerce")

# Use a deterministic subset: top-history markets over the first 120 unique dates.
top_markets = raw_poly.groupby("market_id")["date"].nunique().sort_values(ascending=False).head(40).index
poly_subset = raw_poly[raw_poly["market_id"].isin(top_markets)].copy()
first_dates = np.sort(poly_subset["date"].dropna().unique())[:90]
poly_subset = poly_subset[poly_subset["date"].isin(first_dates)].copy()

poly_df = enrich_snapshot(poly_subset)
# For the daily panel, the observed daily Yes price is in `price`; use it as the implied price series.
poly_df["implied_0"] = pd.to_numeric(poly_df["price"], errors="coerce")
poly_df["id"] = poly_df["market_id"]
poly_df["createdAt"] = pd.to_datetime(poly_df["date"], utc=True, errors="coerce")

subset_summary = {
    "rows": len(poly_df),
    "markets": int(poly_df["market_id"].nunique()),
    "dates": int(poly_df["createdAt"].dt.date.nunique()),
    "date_min": str(poly_df["createdAt"].min().date()),
    "date_max": str(poly_df["createdAt"].max().date()),
}
display(pd.DataFrame([subset_summary]))

In [ ]:
poly_out = OUT / "polymarket_mispricing"
poly_cfg = MispricingConfig(
    min_train_rows=100,
    n_opt_iter=50,
    gamma=2.0,
    kappa=0.02,
    w_contract_max=0.05,
    w_event_max=0.20,
    liq_budget=2.5,
)
poly_result = run_walk_forward(poly_df, cfg=poly_cfg, out_dir=poly_out)
plot_run(poly_result, out_dir=poly_out)

pnl = poly_result["pnl"].copy()
weights = poly_result["weights"].copy()
summary_by_split = pd.DataFrame(poly_result["summary"].get("pnl_by_split", {})).T
summary_core = {
    "n_periods": len(pnl),
    "mean_objective": poly_result["summary"].get("mean_objective"),
    "mean_mse": poly_result["summary"].get("mean_mse"),
    "mean_turnover": poly_result["summary"].get("mean_turnover"),
    "total_next_step_pnl": float(pnl["pnl_realized_next_step"].sum()) if len(pnl) else np.nan,
}

display(pd.DataFrame([summary_core]).round(6))
display(summary_by_split.round(6))
print("Saved demo artifacts to:", poly_out)

In [ ]:
if len(pnl):
    pnl_plot = pnl.copy()
    pnl_plot["date"] = pd.to_datetime(pnl_plot["date"])
    pnl_plot = pnl_plot.sort_values("date")
    pnl_plot["cumulative_pnl"] = pnl_plot["pnl_realized_next_step"].cumsum()

    ax = pnl_plot.plot(x="date", y="cumulative_pnl", figsize=(9, 4), title="Polymarket demo: cumulative next-step PnL")
    ax.set_xlabel("date")
    ax.set_ylabel("cumulative PnL proxy")
    plt.tight_layout()
    plt.show()

if len(weights):
    constraint_summary = weights.groupby("date").agg(
        gross_weight=("weight", "sum"),
        max_contract_weight=("weight", "max"),
        n_positions=("weight", lambda s: int((s > 1e-8).sum())),
        mean_edge=("edge", "mean"),
    )
    display(constraint_summary.describe().round(4))

    ax = weights.plot.scatter(x="edge", y="weight", s=8, alpha=0.35, figsize=(7, 4), title="Position size vs estimated edge")
    plt.tight_layout()
    plt.show()

    top_positions = weights.sort_values("weight", ascending=False).head(10)
    display(top_positions[["date", "id", "weight", "p_hat", "c_exec", "implied", "next_implied", "edge"]].round(5))

## What this demo establishes

- The equity section reproduces the **dynamic rolling optimizer** pattern from the project notebooks using the checked-in S&P 500 data.
- The Polymarket section exercises the production-style modules: loader/enrichment, feature generation, baseline model, executable-price approximation, projected-gradient optimizer, walk-forward loop, saved outputs, and diagnostics.
- For full-size reproduction commands, use the top-level README.